# ProcureSight AI — Procurement Data Audit

## Purpose

This notebook examines the raw procurement dataset before analysis or machine learning. It validates the file structure, data types, missing values, duplicates, date quality, and potential business risks.

## Dataset

Source: Procurement KPI Analysis Dataset from Kaggle  
Raw file: `data/raw/procurement_kpi.csv`

In [11]:
from pathlib import Path

# Find the project root whether the notebook starts from the
# project folder or the notebooks folder.
PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "procurement_kpi.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path: {DATA_PATH}")
print(f"File exists: {DATA_PATH.exists()}")

Project root: c:\Users\Alsae\OneDrive\Desktop\procuresight-ai
Dataset path: c:\Users\Alsae\OneDrive\Desktop\procuresight-ai\data\raw\procurement_kpi.csv
File exists: True


## 1. Load and Preview the Raw Dataset

DuckDB reads the CSV directly and automatically identifies its column types.  
The result is then converted into a pandas DataFrame for interactive exploration.

At this stage, the raw data is only being read—it is not modified.

In [12]:
import duckdb
import pandas as pd

# Read the untouched CSV using DuckDB, then convert it to pandas.
df = duckdb.read_csv(str(DATA_PATH)).df()

print(f"DuckDB version: {duckdb.__version__}")
print(f"pandas version: {pd.__version__}")
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

df.head()

DuckDB version: 1.5.5
pandas version: 3.0.5
Dataset shape: 777 rows × 11 columns


,PO_ID,Supplier,Order_Date,Delivery_Date,Item_Category,Order_Status,Quantity,Unit_Price,Negotiated_Price,Defective_Units,Compliance
0,PO-00001,Alpha_Inc,2023-10-17,2023-10-25,Office Supplies,Cancelled,1176,20.13,17.81,NaN,True
1,PO-00002,Delta_Logistics,2022-04-25,2022-05-05,Office Supplies,Delivered,1509,39.32,37.34,235.0,True
2,PO-00003,Gamma_Co,2022-01-26,2022-02-15,MRO,Delivered,910,95.51,92.26,41.0,True
3,PO-00004,Beta_Supplies,2022-10-09,2022-10-28,Packaging,Delivered,1344,99.85,95.52,112.0,True
4,PO-00005,Delta_Logistics,2022-09-08,2022-09-20,Raw Materials,Delivered,1180,64.07,60.53,171.0,False


## 2. Dataset Structure and Completeness

This section audits each column's data type, completeness, and number of unique values.

- **Missing count and percentage** identify incomplete fields.
- **Unique values** help distinguish identifiers, categories, and continuous measurements.
- **Data types** confirm whether dates, numbers, categories, and Boolean values were interpreted correctly.

In [13]:
schema_audit = pd.DataFrame(
    {
        "Column": df.columns,
        "Data Type": [str(dtype) for dtype in df.dtypes],
        "Non-Null Values": df.notna().sum().values,
        "Missing Values": df.isna().sum().values,
        "Missing %": (df.isna().mean().values * 100).round(2),
        "Unique Values": df.nunique(dropna=True).values,
    }
)

schema_audit

,Column,Data Type,Non-Null Values,Missing Values,Missing %,Unique Values
0,PO_ID,str,777,0,0.0,777
1,Supplier,str,777,0,0.0,5
2,Order_Date,datetime64[us],777,0,0.0,476
3,Delivery_Date,datetime64[us],690,87,11.2,449
4,Item_Category,str,777,0,0.0,5
5,Order_Status,str,777,0,0.0,4
6,Quantity,int64,777,0,0.0,623
7,Unit_Price,float64,777,0,0.0,747
8,Negotiated_Price,float64,777,0,0.0,752
9,Defective_Units,float64,641,136,17.5,202


### Initial Structure Findings

- The dataset contains **777 purchase orders** and **11 original columns**.
- `PO_ID` contains 777 unique values and can therefore act as the order identifier.
- `Delivery_Date` contains **87 missing values (11.2%)**.
- `Defective_Units` contains **136 missing values (17.5%)**.
- The remaining fields are complete.
- `Order_Date` and `Delivery_Date` were correctly interpreted as dates.
- `Compliance` was correctly converted into a Boolean target.
- Missing values will be investigated by order status before deciding whether to retain, impute, or exclude them.

## 3. Missing Values by Order Status

We compare missing delivery dates and defective-unit counts across order statuses.

Percentages are calculated within each status so that groups of different sizes can be compared fairly.

This check identifies patterns but does not establish why values are missing. No values are filled or removed at this stage.

In [14]:
# Create temporary missing-value flags without changing df.
missing_by_status = (
    df.assign(
        Missing_Delivery=df["Delivery_Date"].isna(),
        Missing_Defects=df["Defective_Units"].isna(),
    )
    .groupby("Order_Status", dropna=False)
    .agg(
        Orders=("PO_ID", "size"),
        Missing_Delivery=("Missing_Delivery", "sum"),
        Missing_Defects=("Missing_Defects", "sum"),
    )
)

# Calculate missing percentages within each order status.
missing_by_status["Missing_Delivery_%"] = (
    100 * missing_by_status["Missing_Delivery"]
    / missing_by_status["Orders"]
).round(2)

missing_by_status["Missing_Defects_%"] = (
    100 * missing_by_status["Missing_Defects"]
    / missing_by_status["Orders"]
).round(2)

missing_by_status = missing_by_status[
    [
        "Orders",
        "Missing_Delivery",
        "Missing_Delivery_%",
        "Missing_Defects",
        "Missing_Defects_%",
    ]
]

missing_by_status

,Orders,Missing_Delivery,Missing_Delivery_%,Missing_Defects,Missing_Defects_%
Order_Status,,,,,
Cancelled,63,8,12.70,9,14.29
Delivered,560,68,12.14,102,18.21
Partially Delivered,73,5,6.85,13,17.81
Pending,81,6,7.41,12,14.81


### Missing-Value Findings and Preliminary Decisions

- Missing delivery dates and defect counts occur across all four order statuses.
- Among delivered orders, 68 (12.14%) lack a delivery date and 102 (18.21%) lack a defect count.
- Order status alone does not explain the missing values.
- Many pending orders contain delivery dates; we must clarify whether these represent planned or actual delivery dates before calculating delivery-performance KPIs.

Preliminary handling:
- Preserve all records in the raw dataset.
- Do not replace missing defect counts with zero: unknown does not mean defect-free.
- Do not invent missing delivery dates.
- Report the number of usable records alongside KPIs that depend on these fields.
- Decide model-specific missing-value handling later, using training data only.

## 4. Duplicate and Business-Rule Checks

We check for duplicate records, repeated purchase-order identifiers,
and values that violate basic procurement rules.

Flagged records will be reviewed before any correction or exclusion.
Missing values are assessed separately and are not treated as zero.

In [15]:
# Each condition marks rows requiring review.
quality_flags = pd.DataFrame(
    {
        "Duplicate row": df.duplicated(keep=False),
        "Repeated PO_ID": df["PO_ID"].duplicated(keep=False),
        "Delivery before order": (
            df["Delivery_Date"].notna()
            & df["Order_Date"].notna()
            & (df["Delivery_Date"] < df["Order_Date"])
        ),
        "Non-positive quantity": df["Quantity"] <= 0,
        "Non-positive unit price": df["Unit_Price"] <= 0,
        "Non-positive negotiated price": df["Negotiated_Price"] <= 0,
        "Negative defective units": df["Defective_Units"] < 0,
        "Defects exceed quantity": (
            df["Defective_Units"] > df["Quantity"]
        ),
        "Fractional defective units": (
            df["Defective_Units"].notna()
            & (df["Defective_Units"] % 1 != 0)
        ),
        "Negotiated price exceeds unit price": (
            df["Negotiated_Price"] > df["Unit_Price"]
        ),
    },
    index=df.index,
).fillna(False)

quality_summary = (
    quality_flags.sum()
    .rename_axis("Check")
    .reset_index(name="Flagged Rows")
)

quality_summary

,Check,Flagged Rows
0,Duplicate row,0
1,Repeated PO_ID,0
2,Delivery before order,1
3,Non-positive quantity,0
4,Non-positive unit price,0
5,Non-positive negotiated price,0
6,Negative defective units,0
7,Defects exceed quantity,0
8,Fractional defective units,0
9,Negotiated price exceeds unit price,0


In [16]:
# Copy the flagged record for inspection; leave df unchanged.
date_issue = df.loc[
    quality_flags["Delivery before order"]
].copy()

date_issue["Recorded_Duration_Days"] = (
    date_issue["Delivery_Date"] - date_issue["Order_Date"]
).dt.days

date_issue[
    [
        "PO_ID",
        "Supplier",
        "Order_Status",
        "Order_Date",
        "Delivery_Date",
        "Recorded_Duration_Days",
    ]
]

,PO_ID,Supplier,Order_Status,Order_Date,Delivery_Date,Recorded_Duration_Days
100,PO-00101,Alpha_Inc,Delivered,2022-02-27,2022-02-22,-5


### Date Consistency Finding

Purchase order `PO-00101` (Alpha_Inc, Delivered) has an order date
of 2022-02-27 and a delivery date of 2022-02-22, producing a
recorded duration of -5 days.

The source does not establish which date is incorrect.

Handling decision:
- Preserve the original record and dates.
- Flag the record as having an inconsistent date sequence.
- Exclude its duration from delivery-duration calculations.
- Retain the order for other analyses where its fields remain usable.
- Do not swap dates or convert the negative duration to a positive value.

## 5. Compliance Target Distribution

We count compliant and non-compliant orders and calculate their
percentage of the dataset.

This establishes the class balance before modelling. Accuracy alone
can be misleading when one outcome is much more common than the other.

In [17]:
# Translate Boolean values into readable labels for this summary.
compliance_labels = df["Compliance"].map(
    {True: "Compliant", False: "Non-compliant"}
)

compliance_summary = (
    compliance_labels.value_counts(dropna=False)
    .rename_axis("Compliance Status")
    .reset_index(name="Orders")
)

compliance_summary["Share %"] = (
    100 * compliance_summary["Orders"] / len(df)
).round(2)

compliance_summary

,Compliance Status,Orders,Share %
0,Compliant,640,82.37
1,Non-compliant,137,17.63


### Class Balance Findings

- Compliant: 640 orders (82.37%).
- Non-compliant: 137 orders (17.63%).
- The dataset contains substantially more compliant than non-compliant orders.
- Predicting every order as compliant would achieve 82.37% accuracy
  on this dataset, but detect none of the non-compliant orders.
- Model evaluation will therefore include precision, recall, and F1
  for the non-compliant class, alongside a simple baseline model.
- For risk modelling, we will explicitly encode non-compliant as 1
  and compliant as 0, so higher predicted probabilities represent risk.
- The original Compliance column remains unchanged during this audit.

## 6. Time Coverage and Monthly Order Counts

We inspect the order-date range and monthly order volumes before
designing a chronological training and test split.

Monthly non-compliance counts help identify periods with limited
examples. Missing calendar months are included explicitly.

No training or test split is created at this stage.

In [18]:
print(f"First order: {df['Order_Date'].min():%Y-%m-%d}")
print(f"Last order:  {df['Order_Date'].max():%Y-%m-%d}")

# Build a temporary summary without changing the original data.
monthly_orders = (
    df.assign(
        Order_Month=df["Order_Date"].dt.to_period("M"),
        Non_Compliant=df["Compliance"].eq(False),
    )
    .groupby("Order_Month")
    .agg(
        Orders=("PO_ID", "size"),
        Non_Compliant_Orders=("Non_Compliant", "sum"),
    )
)

# Include every calendar month, even months with no orders.
all_months = pd.period_range(
    start=df["Order_Date"].min(),
    end=df["Order_Date"].max(),
    freq="M",
)

monthly_orders = monthly_orders.reindex(all_months, fill_value=0)
monthly_orders.index.name = "Order Month"

monthly_orders

First order: 2022-01-01
Last order:  2024-01-01


,Orders,Non_Compliant_Orders
Order Month,,
2022-01,31,1
2022-02,27,3
2022-03,41,5
2022-04,35,4
2022-05,25,3
2022-06,34,3
2022-07,27,6
2022-08,43,9
2022-09,43,11


### Monthly Order Volume and Compliance

The stacked bars show total orders per month, separated into compliant
and non-compliant orders.

These are counts, not non-compliance rates: a larger non-compliant
segment can reflect a higher total order volume.

January 2024 contains only two orders and should not be interpreted
as evidence of a full-month decline.

In [19]:
import plotly.express as px

# Prepare a separate table for plotting.
monthly_plot = monthly_orders.reset_index()
monthly_plot["Order Month"] = monthly_plot["Order Month"].astype(str)

# Total orders already includes non-compliant orders.
monthly_plot["Compliant"] = (
    monthly_plot["Orders"] - monthly_plot["Non_Compliant_Orders"]
)

monthly_plot = monthly_plot.rename(
    columns={"Non_Compliant_Orders": "Non-compliant"}
)

fig = px.bar(
    monthly_plot,
    x="Order Month",
    y=["Compliant", "Non-compliant"],
    barmode="stack",
    title="Monthly Purchase Orders by Compliance Status",
    labels={
        "value": "Number of orders",
        "variable": "Compliance status",
    },
    color_discrete_map={
        "Compliant": "#2563EB",
        "Non-compliant": "#F97316",
    },
    template="plotly_white",
)

fig.update_layout(
    xaxis_title="Order month",
    yaxis_title="Number of orders",
    legend_title_text="Compliance status",
    height=480,
)

fig.update_xaxes(type="category", tickangle=-45)
fig.update_yaxes(rangemode="tozero")

fig.show()

### Monthly Non-Compliance Rate

The non-compliance rate is the number of non-compliant orders divided
by all orders in that month.

The dashed line shows the overall dataset rate, weighted by order count.
Monthly fluctuations are descriptive and do not establish their causes.

January 2024 contains only two orders. Its observed 0% rate is based
on very limited data and should not be interpreted as reliable evidence
of low risk.

In [20]:
rate_plot = monthly_orders.reset_index()
rate_plot["Order Month"] = rate_plot["Order Month"].astype(str)

# A month with no orders has an undefined rate, not a 0% rate.
rate_plot["Non-compliance rate"] = (
    rate_plot["Non_Compliant_Orders"]
    / rate_plot["Orders"].where(rate_plot["Orders"] > 0)
)

overall_rate = df["Compliance"].eq(False).mean()

fig = px.line(
    rate_plot,
    x="Order Month",
    y="Non-compliance rate",
    markers=True,
    title="Monthly Non-Compliance Rate",
    hover_data={
        "Non-compliance rate": ":.2%",
        "Orders": True,
        "Non_Compliant_Orders": True,
    },
    template="plotly_white",
)

fig.update_traces(line_color="#F97316", connectgaps=False)

fig.add_hline(
    y=overall_rate,
    line_dash="dash",
    line_color="#64748B",
    annotation_text=f"Overall rate: {overall_rate:.2%}",
    annotation_position="top left",
)

fig.add_annotation(
    x="2024-01",
    y=0,
    text="Only 2 orders",
    showarrow=True,
    ax=-65,
    ay=-45,
)

fig.update_layout(height=480)
fig.update_xaxes(type="category", tickangle=-45)
fig.update_yaxes(
    title="Non-compliant share of orders",
    tickformat=".0%",
    rangemode="tozero",
)

fig.show()

### Monthly Rate Findings

- The overall non-compliance rate is 17.63%.
- June 2023 recorded 9 non-compliant orders out of 26 (34.62%).
- October 2023 recorded 10 out of 29 (34.48%).
- Monthly percentages fluctuate, with relatively small order counts.
- January 2024 contains only two orders; its 0% rate does not
  establish that procurement risk improved.
- These observations do not establish seasonality or explain
  the causes of non-compliance.
- Future model evaluation should preserve chronological order.
  Split dates must not be chosen to obtain favourable test results.

## 7. Data Preparation Plan

### Preserve the Source
- Keep the original CSV unchanged in `data/raw`.
- Create a separate working dataset for preparation.
- Retain `PO_ID` for tracing records, but exclude it from model inputs.

### Handle Missing and Inconsistent Data
- Keep missing delivery dates and defect counts as unknown.
- Do not replace missing defect counts with zero.
- Flag the inconsistent date sequence in `PO-00101`.
- Exclude invalid or missing durations from duration calculations.
- Report usable-record coverage alongside affected KPIs.

### Clarify Business Definitions
- Confirm whether `Delivery_Date` represents planned or actual delivery.
- Clarify what the `Compliance` label measures.
- Confirm currency and price definitions before reporting monetary KPIs.
- Do not calculate on-time delivery without a promised delivery date.

### Prepare for Modelling
- Encode non-compliance as 1 and compliance as 0.
- Use only information available when a new order is assessed.
- Exclude the current order's delivery date, defect count, and final
  order status from pre-order prediction.
- Use historical supplier outcomes only when their availability
  before the prediction date can be established.
- Fit preprocessing rules on training data only.
- Use chronological evaluation and compare against simple baselines.
- Evaluate missed risks, false alarms, and probability reliability.

### Next Phase
Implement documented preparation rules, then build supplier analytics
with supporting charts and clearly stated limitations.